# NAE optimization
This file sees how far we can push pyAQSC's aspect ratio

In [1]:
import importlib
import aqsc
import time
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.random as jrd
import optax
import optax.tree
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")
from shared import *
from jax import grad

In [2]:
i_test = 1
# eq_fin = jnp.load(f'pass_2/best_x.npy', allow_pickle=True).item()
fitness = jnp.load(f'./fitness.npy')
population = jnp.load(f'./population.npy')
x_fin = population[jnp.argmin(fitness)]

In [3]:
min(fitness)

Array(1.91239052, dtype=float64)

In [4]:
eq_test = solve_order_6(x_fin, padded=True)

Default interpolation method: cubic
Solving for \bar{\iota}_0 self-consistently.
Evaluating order 3 4
Evaluating order 5 6


# Optimization

In [5]:
fun = lambda x: objective(x, w_aspect, w_anisotropy, w_iota, w_p, 10, full_mode=False, padded=True)

In [6]:
print('final objective value in the globals tage', fun(x_fin))

Default interpolation method: cubic
Solving for \bar{\iota}_0 self-consistently.
Evaluating order 3 4
Evaluating order 5 6
final objective value in the globals tage 2.1571395312788075


In [7]:
eq = eq_test
anisotropy = rms_anisotropy(eq)

psi_crit, _, _ = eq.get_psi_crit()
eps_crit_val = jnp.sqrt(psi_crit)
eps_conv_val = eps_conv(eq)
eps = jnp.minimum(eps_conv_val, eps_crit_val)
aspect = eq.aspect_ratio_eps(eps)

# Iota
iota_a = jnp.real(eq.iota.eval(psi=0, chi=0, phi=0))

# Effective field strength at the edge
# B_denom_20 = jnp.real(in_dict['B20'])|
# Effective pressure at the edge
p20_avg = jnp.real(jnp.average(eq.p_perp[2][0].content))
p00_avg = jnp.real(jnp.average(eq.p_perp[0][0].content))
p_edge_eff = (p00_avg + p20_avg*eps**2)
p_axis_eff = p00_avg
# # beta_edge eff
# beta_axis_eff = p00_avg*B_denom_0
# beta_edge_eff = p_edge_eff*B_denom_edge_eff
# B2_eff = 1/(B_denom_0 + B_denom_20 * eps**2)
term1 = w_aspect * (
    jnp.maximum(aspect - target_aspect, 0) / target_aspect
)**2
term2 = w_anisotropy * (
    jnp.maximum(anisotropy - target_anisotropy, 0) / target_anisotropy
)**2
term3 = w_iota * (
    jnp.maximum(jnp.abs(target_iota) - jnp.abs(iota_a), 0) / target_iota
)**2
# term4 = w_p * (
#     jnp.maximum(p20_avg, 0) / p20_avg
# )**2
term4 = w_p * (
    jnp.maximum(p_edge_eff - p_axis_eff, 0) / p_axis_eff
)**2
out = term1 + term2 + term3 + term4
print('term1', term1)
print('term2', term2)
print('term3', term3)
print('term4', term4)
print('eps_crit', eps_crit_val,)
print('eps_conv', eps_conv_val,)
print('aspect', aspect,)
print('anisotropy', anisotropy,)
print('p_edge_eff', p_edge_eff,)
print('p_axis_eff', p_axis_eff,)
print('iota_a', iota_a,)

term1 2.1571395312788075
term2 0.0
term3 0.0
term4 0.0
eps_crit 0.019670324459802126
eps_conv 0.01873716567895077
aspect 39.99006452648584
anisotropy 0.009533695763720698
p_edge_eff 0.006663566518806374
p_axis_eff 0.006783190144439473
iota_a 0.17313383561803164


In [8]:
# Crashes kernel

# # Define optimizer
# lr = 1e-1
# opt = optax.scale_by_lbfgs()
# update = jit(opt.update)

# # Define objective
# niter = 1000

# # Initialize optimization
# w_opt = x_fin
# state = opt.init(w_opt)

# # Run optimization
# for i in range(niter):
#     v, g = jax.value_and_grad(fn)(w)
#     if i%10 == 0:
#         print(f'Iteration: {i}, Value:{v:.2e}')
#     u, state = update(g, state, w)
#     w = w - lr * u

# print(f'Final value: {fun(w):.2e}')

In [9]:
obj_wrapped = jit(fun)
jac = jit(grad(fun))

In [10]:
def jac_safe(x_flat):
    # jac = jit(grad(fun)) only takes x; scipy.minimize calls jac(x).
    out = jac(x_flat)
    return jnp.nan_to_num(out, nan=0., posinf=0., neginf=0.)


In [11]:
%%time
# Compiling val
print('Init val', fun(x_flat_init))

Default interpolation method: cubic
Solving for \bar{\iota}_0 self-consistently.
Evaluating order 3 4
Evaluating order 5 6
Init val 2183.425237729622
CPU times: user 1min 10s, sys: 12 s, total: 1min 21s
Wall time: 59.9 s


In [12]:
class PrintEvery10:
    def __init__(self, fun):
        self.fun = fun
        self.n = 0

    def __call__(self, xk):
        if self.n % 2 == 0:
            print(f"iter {self.n:5d} | f(x) = {self.fun(xk):.6e}")
        self.n += 1

callback_aspect = PrintEvery10(fun=lambda x: aspect_conv(solve_order_6(x)))

In [ ]:
%%time
# Compiling jac = jit(grad(fun)). Expect a long first compile (order-6 reverse
# mode ~2.5M jaxpr eqns); it is not hung. Restart kernel after shared.py
# changes (stop_gradient(psi_crit), no nested @jit helpers).
print('Init jac', jac(x_flat_init))


Default interpolation method: cubic
Solving for \bar{\iota}_0 self-consistently.
Evaluating order 3 4
Evaluating order 5 6


/scratch/miniconda3/envs/desc/lib/python3.12/site-packages/jax/_src/lax/lax.py:5377: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


In [ ]:
# x_flat_init + np.random.random(len(res.x))
def minimize_step(x_init, i:int, method='L-BFGS-B'):
    time1 = time.time()
    res = minimize(
        fun=obj_wrapped, 
        x0=x_init, 
        jac=jac_safe, 
        # callback=callback_aspect,
        method=method,  
        # method='Nelder-Mead',
        # method='CG', # May work? 
        # method='trust-constr',
        options={
            'maxiter':200,
            # 'maxls':50,
        }
    )
    time2 = time.time()
    return res, time2-time1